# Nested five-fold ε selection on O-HAZE

This notebook evaluates an inference-only ε-selection procedure for the three fixed Stage 4 checkpoints. It holds the ASM blend at 0.08, uses the observed-hazy ASM formulation, and evaluates all 45 O-HAZE pairs at 512×512 with radiometric preprocessing. Each pair remains in one image-level fold.

For each outer fold, choose one shared ε using mean PSNR across the other 36 images and all three seeds; use mean SSIM only as a tie-breaker. Score that ε once on the nine held-out images. Compare out-of-fold results with fixed ε=0.10 using paired, image-clustered bootstrap intervals. This estimates the selection procedure; it does not produce an independently confirmed final ε.

**Interpretation limit:** the manuscript already reports O-HAZE results at ε=0.10. The fold-wise sweep keeps each outer image out of its fold's ε choice, but this is not a wholly untouched external evaluation if prior O-HAZE outcomes influenced parameter decisions. Do not describe it as independent external confirmation. It also cannot validate 0.08 as a training blend: checkpoints remain fixed.

The paths below follow the sweep-analysis.ipynb reference and the saved O-HAZE run manifest. Edit them for another runtime.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import time

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

# Kaggle paths used by the reference notebook and prior O-HAZE evaluation.
PACKAGE = Path("/kaggle/input/datasets/randomguy28/reviewer-test-packge/reviewer_test_package")
OHAZE_ROOT = Path("/kaggle/input/datasets/philiphofmann/o-haze/O-HAZY")
HAZY_DIR = OHAZE_ROOT / "hazy"
GT_DIR = OHAZE_ROOT / "GT"
OUTPUT_ROOT = Path("/kaggle/working/nested_ohaze_epsilon_cv")

SCRIPT_ROOT = PACKAGE / "code"
CHECKPOINTS = {
    seed: PACKAGE / f"models/full/seed{seed}/stage4_best.keras"
    for seed in (17, 43, 73)
}
EPSILONS = (0.01, 0.05, 0.10, 0.20, 0.50)
BLEND = 0.08
IMAGE_SIZE = 512
PREPROCESSING = "radiometric"
N_FOLDS = 5
FOLD_SEED = 42
BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_SEED = 20261014
BASELINE_EPSILON = 0.10

assert PACKAGE.is_dir(), f"Check PACKAGE path: {PACKAGE}"
assert HAZY_DIR.is_dir(), f"Check hazy directory: {HAZY_DIR}"
assert GT_DIR.is_dir(), f"Check ground-truth directory: {GT_DIR}"
assert all(path.is_file() for path in CHECKPOINTS.values()), "Missing Stage 4 checkpoint."

sys.path.insert(0, str(SCRIPT_ROOT))
from eca_ldnet.data import get_gt_path, load_image_pair
from eca_ldnet.layers import CUSTOM_OBJECTS

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

image_exts = {".png", ".jpg", ".jpeg", ".bmp"}
pair_records = []
unmatched = []
for hazy_path in sorted(p for p in HAZY_DIR.iterdir() if p.suffix.lower() in image_exts):
    gt_path = get_gt_path(hazy_path.name, str(GT_DIR))
    if gt_path is None:
        unmatched.append(hazy_path.name)
    else:
        pair_records.append({
            "image": hazy_path.name,
            "hazy_path": str(hazy_path),
            "clean_path": str(gt_path),
        })

missing = [
    row[key]
    for row in pair_records
    for key in ("hazy_path", "clean_path")
    if not Path(row[key]).is_file()
]
print("Matched O-HAZE pairs:", len(pair_records))
print("Unmatched hazy images:", unmatched)
print("Missing files:", missing[:5])
print("Checkpoints:", {seed: path.name for seed, path in CHECKPOINTS.items()})
assert len(pair_records) == 45, f"Expected 45 O-HAZE pairs; found {len(pair_records)}."
assert not unmatched and not missing, "Fix pair paths before running."
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## Run fixed-checkpoint ε sweep

This cell makes one forward pass per image and checkpoint, then scores five ε values from the saved heads. It writes seed-level and combined per-image CSVs. No training occurs.

In [ ]:
all_rows = []
checkpoint_hashes = {seed: sha256_file(path) for seed, path in CHECKPOINTS.items()}
started = time.perf_counter()

for seed, checkpoint in CHECKPOINTS.items():
    print(f"Loading seed {seed}: {checkpoint}")
    model = tf.keras.models.load_model(
        str(checkpoint), custom_objects=CUSTOM_OBJECTS, compile=False
    )
    if model.input_shape[1] is not None:
        assert model.input_shape[1] == IMAGE_SIZE, (
            f"Checkpoint expects height {model.input_shape[1]}, not {IMAGE_SIZE}."
        )
    heads = tf.keras.Model(
        model.input,
        [
            model.get_layer("raw_sigmoid").output,
            model.get_layer("atm_light").output,
            model.get_layer("t_full").output,
        ],
    )
    seed_rows = []

    for index, pair in enumerate(pair_records, start=1):
        hazy, clean = load_image_pair(
            pair["hazy_path"],
            pair["clean_path"],
            size=IMAGE_SIZE,
            preprocessing=PREPROCESSING,
        )
        if hazy is None or clean is None:
            raise RuntimeError(f"Could not read pair: {pair['image']}")

        raw_b, atm_b, transmission_b = heads.predict(hazy[None, ...], verbose=0)
        raw = raw_b[0]
        atm = atm_b[0].reshape(1, 1, 3)
        transmission = transmission_b[0]

        for epsilon in EPSILONS:
            asm_estimate = np.clip(
                (hazy - atm) / (transmission + epsilon) + atm, 0.0, 1.0
            )
            prediction = np.clip(
                raw * (1.0 - BLEND) + asm_estimate * BLEND, 0.0, 1.0
            ).astype(np.float32)
            psnr = float(
                tf.image.psnr(prediction[None, ...], clean[None, ...], 1.0)
                .numpy().reshape(-1)[0]
            )
            ssim = float(
                tf.image.ssim(prediction[None, ...], clean[None, ...], 1.0)
                .numpy().reshape(-1)[0]
            )
            seed_rows.append({
                "seed": seed,
                "image": pair["image"],
                "epsilon": epsilon,
                "blend": BLEND,
                "psnr": psnr,
                "ssim": ssim,
            })

        if index % 5 == 0 or index == len(pair_records):
            print(f"  seed {seed}: {index}/{len(pair_records)} images")

    seed_frame = pd.DataFrame(seed_rows)
    seed_frame.to_csv(OUTPUT_ROOT / f"seed{seed}_per_image.csv", index=False)
    all_rows.extend(seed_rows)
    del heads, model
    tf.keras.backend.clear_session()

metrics = pd.DataFrame(all_rows)
expected_rows = len(pair_records) * len(CHECKPOINTS) * len(EPSILONS)
assert len(metrics) == expected_rows, f"Expected {expected_rows} metric rows; got {len(metrics)}."
metrics.to_csv(OUTPUT_ROOT / "all_seeds_per_image.csv", index=False)
print(f"Saved {len(metrics)} rows to {OUTPUT_ROOT}")
print(f"Elapsed: {(time.perf_counter() - started) / 60:.1f} min")


## Nested five-fold selection

Fold assignment is deterministic and image-disjoint. Each outer fold has nine images; the remaining 36 select one shared ε by mean PSNR across images and seeds. The outer fold is used only for scoring the selected ε and the fixed ε=0.10 reference. Do not alter the selection rule after inspecting these results.

In [ ]:
metrics_path = OUTPUT_ROOT / "all_seeds_per_image.csv"
metrics = pd.read_csv(metrics_path)
assert set(metrics["epsilon"].unique()) == set(EPSILONS)
assert set(metrics["seed"].unique()) == set(CHECKPOINTS)
assert metrics.groupby(["seed", "image", "epsilon"]).size().eq(1).all()

images = np.array(sorted(metrics["image"].unique()))
assert len(images) == 45
rng = np.random.default_rng(FOLD_SEED)
shuffled = images.copy()
rng.shuffle(shuffled)
fold_images = np.array_split(shuffled, N_FOLDS)
image_to_fold = {
    image: fold
    for fold, fold_group in enumerate(fold_images, start=1)
    for image in fold_group
}
metrics["outer_fold"] = metrics["image"].map(image_to_fold)

fold_assignments = pd.DataFrame(
    [{"image": image, "outer_fold": image_to_fold[image]} for image in images]
)
fold_assignments.to_csv(OUTPUT_ROOT / "fold_assignments.csv", index=False)
assert fold_assignments.groupby("outer_fold")["image"].nunique().eq(9).all()

fold_rows = []
oof_rows = []
for fold in range(1, N_FOLDS + 1):
    inner = metrics.loc[metrics["outer_fold"] != fold]
    outer = metrics.loc[metrics["outer_fold"] == fold]

    inner_means = (
        inner.groupby("epsilon", as_index=False)
        .agg(inner_psnr=("psnr", "mean"), inner_ssim=("ssim", "mean"))
        .sort_values(
            ["inner_psnr", "inner_ssim", "epsilon"],
            ascending=[False, False, True],
        )
    )
    selected_epsilon = float(inner_means.iloc[0]["epsilon"])

    selected = outer.loc[outer["epsilon"] == selected_epsilon, [
        "image", "seed", "psnr", "ssim"
    ]].rename(columns={"psnr": "selected_psnr", "ssim": "selected_ssim"})
    baseline = outer.loc[outer["epsilon"] == BASELINE_EPSILON, [
        "image", "seed", "psnr", "ssim"
    ]].rename(columns={"psnr": "baseline_psnr", "ssim": "baseline_ssim"})
    fold_oof = selected.merge(baseline, on=["image", "seed"], validate="one_to_one")
    fold_oof["outer_fold"] = fold
    fold_oof["selected_epsilon"] = selected_epsilon
    fold_oof["delta_psnr"] = fold_oof["selected_psnr"] - fold_oof["baseline_psnr"]
    fold_oof["delta_ssim"] = fold_oof["selected_ssim"] - fold_oof["baseline_ssim"]
    oof_rows.extend(fold_oof.to_dict("records"))

    selected_outer = fold_oof[["selected_psnr", "selected_ssim"]].mean()
    baseline_outer = fold_oof[["baseline_psnr", "baseline_ssim"]].mean()
    fold_rows.append({
        "outer_fold": fold,
        "n_inner_images": inner["image"].nunique(),
        "n_outer_images": outer["image"].nunique(),
        "selected_epsilon": selected_epsilon,
        "inner_psnr_at_selected": float(inner_means.iloc[0]["inner_psnr"]),
        "inner_ssim_at_selected": float(inner_means.iloc[0]["inner_ssim"]),
        "outer_selected_psnr": float(selected_outer["selected_psnr"]),
        "outer_baseline_psnr_eps_0.10": float(baseline_outer["baseline_psnr"]),
        "outer_delta_psnr": float(selected_outer["selected_psnr"] - baseline_outer["baseline_psnr"]),
        "outer_selected_ssim": float(selected_outer["selected_ssim"]),
        "outer_baseline_ssim_eps_0.10": float(baseline_outer["baseline_ssim"]),
        "outer_delta_ssim": float(selected_outer["selected_ssim"] - baseline_outer["baseline_ssim"]),
    })

fold_results = pd.DataFrame(fold_rows)
oof = pd.DataFrame(oof_rows)
fold_results.to_csv(OUTPUT_ROOT / "fold_selection_and_scores.csv", index=False)
oof.to_csv(OUTPUT_ROOT / "out_of_fold_per_image.csv", index=False)
display(fold_results.round(5))


In [ ]:
# Paired bootstrap resamples images, keeping each image's three seed results together.
image_means = (
    oof.groupby("image", as_index=False)
    .agg(
        selected_psnr=("selected_psnr", "mean"),
        baseline_psnr=("baseline_psnr", "mean"),
        selected_ssim=("selected_ssim", "mean"),
        baseline_ssim=("baseline_ssim", "mean"),
    )
)
image_means["delta_psnr"] = image_means["selected_psnr"] - image_means["baseline_psnr"]
image_means["delta_ssim"] = image_means["selected_ssim"] - image_means["baseline_ssim"]
boot_rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_summary = {}

for metric in ("psnr", "ssim"):
    values = image_means[f"delta_{metric}"].to_numpy()
    sample_ids = boot_rng.integers(
        0, len(values), size=(BOOTSTRAP_SAMPLES, len(values))
    )
    boot_means = values[sample_ids].mean(axis=1)
    bootstrap_summary[metric] = {
        "mean_delta": float(values.mean()),
        "ci95_low": float(np.quantile(boot_means, 0.025)),
        "ci95_high": float(np.quantile(boot_means, 0.975)),
    }

summary = pd.DataFrame([
    {"metric": metric, **values}
    for metric, values in bootstrap_summary.items()
])
summary.to_csv(OUTPUT_ROOT / "out_of_fold_paired_bootstrap.csv", index=False)
display(summary)
print("Fold ε selections:", fold_results["selected_epsilon"].value_counts().sort_index().to_dict())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].bar(
    fold_results["outer_fold"].astype(str),
    fold_results["selected_epsilon"],
    color="#4472C4",
)
axes[0].set(
    xlabel="Outer fold",
    ylabel="ε selected on other 36 images",
    title="Fold-wise ε selection",
    ylim=(0, max(EPSILONS) * 1.15),
)
axes[0].grid(axis="y", alpha=0.25)

labels = ["PSNR (dB)", "SSIM"]
keys = ["psnr", "ssim"]
means = [bootstrap_summary[key]["mean_delta"] for key in keys]
low = [bootstrap_summary[key]["ci95_low"] for key in keys]
high = [bootstrap_summary[key]["ci95_high"] for key in keys]
lower_err = [mean - lo for mean, lo in zip(means, low)]
upper_err = [hi - mean for mean, hi in zip(means, high)]
axes[1].errorbar(
    labels,
    means,
    yerr=[lower_err, upper_err],
    fmt="o",
    capsize=5,
    color="#C0504D",
)
axes[1].axhline(0, color="black", linewidth=1, linestyle="--")
axes[1].set(
    ylabel="Out-of-fold selected ε minus fixed ε=0.10",
    title="Paired image-cluster bootstrap 95% CI",
)
axes[1].grid(axis="y", alpha=0.25)

fig.tight_layout()
figure_path = OUTPUT_ROOT / "nested_ohaze_epsilon_summary.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", figure_path)

run_manifest = {
    "dataset": "O-HAZE",
    "n_pairs": len(pair_records),
    "image_size": IMAGE_SIZE,
    "preprocessing": PREPROCESSING,
    "source": "observed_hazy",
    "blend_fixed": BLEND,
    "epsilon_grid": list(EPSILONS),
    "seeds": list(CHECKPOINTS),
    "checkpoint_sha256": checkpoint_hashes,
    "outer_folds": N_FOLDS,
    "fold_seed": FOLD_SEED,
    "selection_rule": "One shared epsilon per outer fold; maximize inner mean PSNR across images and seeds; SSIM tie-breaker.",
    "baseline_epsilon": BASELINE_EPSILON,
    "bootstrap_unit": "image, with three seed outcomes kept together",
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "caveat": "O-HAZE at epsilon=0.10 was previously reported; this is not a wholly untouched external evaluation if prior O-HAZE outcomes influenced parameter decisions.",
    "paired_results": bootstrap_summary,
}
manifest_path = OUTPUT_ROOT / "run_manifest.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2))
print("Saved run manifest:", manifest_path)
